Ch 1-6: Array, Strings, Lists, Stacks, Queues, Trees, Graphs, etc-- [Open in Colab](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1-6.ipynb)

Python Basics: [data structures](https://docs.python.org/3/tutorial/datastructures.html), [deque](https://docs.python.org/3/library/collections.html#collections.deque), [heapq](https://docs.python.org/3/library/heapq.html), [string methods](https://docs.python.org/3.4/library/stdtypes.html#string-methods), [itertools](https://docs.python.org/3/library/itertools.html#itertools-recipes), [functools](https://docs.python.org/3/library/functools.html), [match-case](https://www.inspiredpython.com/course/pattern-matching/mastering-structural-pattern-matching), [new python features](https://www.nicholashairs.com/posts/major-changes-between-python-versions/), [python tricks](https://sahandsaba.com/14-more-python-features-and-tricks-you-may-not-know.html)

---

20 == len(list(permutations(range(5), 2))) and len(combinations(range(5), 2)) == 10

* a k-combination of a set S is a subset of k distinct elements of S, and the # of k-combinations is equals to the binomial coefficient, n! / (k! * (n-k)!).
* a k-permutation of a set S is an ordered sequence of k distinct elements of S, and the # of k-permutation of n objects is denoted variously nPk, Pn,k, and P(n,k), and its value is given by n! / (n-k)!.

In [ ]:
# @title import before you begin
from functools import lru_cache, reduce
from itertools import accumulate, islice, chain, count, pairwise, tee
from collections import namedtuple, Counter, defaultdict, deque
from dataclasses import dataclass
from enum import Enum, IntFlag
from typing import Optional, Union
from math import *
from random import *
from sys import maxsize as inf  # infinity: 2**63-1.
import bisect
import more_itertools
import operator

class BTree:
  def __init__(self, key=None, left=None, right=None, value=None):
    self.key, self.left, self.right, self.value_ = key, left, right, value

  @property
  def value(self):
    return self.key if self.value_ is None else self.value_

def renumerate(it, stop=None):
  return (L := list(it)).reverse() or zip(count(stop or len(L), -1), L)

def recap(kv, k, v):
  return (kv[0] + k, kv[1] + v)

def dedupe(iterable):
  seen = set()
  for e in iterable:
    if e not in seen:
      seen.add(e)
      yield e

def maxima(iterable, key=None):
  return minima(iterable, key, inverse=True)

def minima(iterable, key=None, inverse=False):
  iterator = iter(iterable)
  minima = [next(iterator, None)]
  key = key or (lambda e: e)
  min_key = minima and key(minima[0])
  lt_ = operator.gt if inverse else operator.lt
  for e in iterator:
    if lt_((k := key(e)), min_key):
      minima, min_key = [e], k
    elif k == min_key:
      minima.append(e)
  return minima

Cut = namedtuple("Cut", "start stop", defaults=[0, 0])
Cut.off = classmethod(lambda cls, stop: cls(start=0, stop=stop))
Cut.__abs__ = lambda self: abs(self.stop - self.start)
Cut.__int__ = lambda self: self.stop - self.start
Cut.__call__ = lambda self, seq: seq[self.start:self.stop]

Intvl = namedtuple("Intvl", "lend, rend")  # left-, and right-closed ends.
Intvl.__abs__ = lambda self: abs(self.rend+1 - self.lend)

def __repr__(obj):
  m = len(values := list(v for (k, v) in vars(obj).items() if k[0] != "_"))
  n = next((i for i, e in enumerate(reversed(values)) if e is not None), m)
  return f"{obj.__class__.__name__}({repr(values[:m-n])[1:-1]})"

@dataclass
class SNode:
  value: int = None
  next: "SNode" = None
  #
  def __iter__(self):
    L = self
    while L:
      yield L
      L = L.next

  @classmethod
  def from_values(cls, *values):
    L = None
    for value in reversed(values):
      L = cls(value, L)
    return L  # the head node of the list.

  @classmethod
  def pool(cls, *values):
    push = lambda acc, new: setattr(new, "next", acc) or new
    # return deque(accumulate(map(cls, reversed(values)), push), maxlen=1).pop()
    return reduce(push, map(cls, reversed(values)))

class BTree:
  def __init__(self, key=None, left=None, right=None, value=None):
    self.key, self.left, self.right, self.value_ = key, left, right, value

  @property
  def value(self):
    return self.key if self.value_ is None else self.value_

def from_values(cls, values, start=0, stop=None):
  if stop is None:
    stop = len(values)
  if stop - start>0:
    mid = (start+stop-1) // 2  # mid becomes start, when stop-start-2.
    L = cls.from_values(values, start, mid)
    R = cls.from_values(values, mid+1, stop)
    return cls(values[mid], L, R, None)

BTree.__repr__ = __repr__  # yapf:disable
BTree.from_values = classmethod(from_values)

In [ ]:
def symmetric(tree):  # is a binary tree symmetric?
  if tree is None:
    return True, (), ()  # return (outcome, inorder, and preorder).
  L = symmetric(tree.left)
  R = symmetric(tree.right)
  outcome = L[1] == R[1][::-1] and L[2] == R[2]
  in_ = L[1] + (tree.key,) + R[1]
  pre = (tree.key,) + L[2] + R[2]
  return outcome, in_, pre

def is_symmetric(tree):
  return are_symmetric_subtrees(tree.left, tree.right) if tree else True

def are_symmetric_subtrees(L, R):  # symmetric left and right subtrees?
  return (L == R  #
          or (L and R  #
              and L.value == R.value  #
              and are_symmetric_subtrees(L.left, R.right)  #
              and are_symmetric_subtrees(L.right, R.left)))

# tree:   1
#       2   2
#     3       3
#      4     4
left = BTree(2, BTree(3, None, BTree(4)))
right = BTree(2, None, BTree(3, BTree(4)))
b7 = BTree(1, left, right)
assert is_symmetric(b7)
assert symmetric(b7)[0]

In [ ]:
def dfs_tree(tree):
  if tree:
    yield (OP.ENTER, tree)
    yield from chain(dfs(tree.left), dfs(tree.right))
    yield (OP.LEAVE, tree)

def bfs_tree(tree):
  q = deque([tree])
  while q:
    yield (tree := q.popleft())
    for e in (tree.left, tree.right):
      e and q.append(e)

preordered = [1, 2, 3, 4, 2, 3, 4]
assert preordered == [e.key for e_type, e in dfs_tree(b7) if e_type == OP.ENTER]
assert [1, 2, 2, 3, 3, 4, 4] == [tree.key for tree in bfs_tree(b7)]

In [ ]:
# Given a binary tree, find its exterior of left boundary nodes, leaf, and right boundary nodes.
def is_leaf(tree):
  return not any((tree.left, tree.right))

def left_boundary_leaves(tree, exterior):
  if tree:
    if exterior or is_leaf(tree):
      yield tree.value
    yield from left_boundary_leaves(tree.left, exterior)
    yield from right_boundary_leaves(tree.right, False)

def right_boundary_leaves(tree, exterior):
  if tree:
    if exterior or is_leaf(tree):
      yield tree.value
    yield from right_boundary_leaves(tree.right, exterior)
    yield from left_boundary_leaves(tree.left, False)

def exterior(tree):
  if tree:
    yield tree.value
    yield from left_boundary_leaves(tree.left, True)
    yield from reversed(list(right_boundary_leaves(tree.right, True)))

# tree:     a
#         /   \
#       b      f
#     c  e   g  h
#   d       i

d = BTree('d')
c = BTree('c', d)
e = BTree('e')
b = BTree('b', c, e)
a = BTree('a', b, BTree('f', BTree('g', BTree('i')), BTree('h')))

assert ['a', 'b', 'c', 'd', 'e', 'i', 'h', 'f'] == list(exterior(a))

In [ ]:
def acceptable(candidate, j):
  i = candidate[-1]
  k = i ^ j
  return k and (k & k-1) == 0

def gray_code(n, candidate=None, accepted=None):
  candidate = candidate or []
  accepted = accepted or set()
  if len(candidate) == 2**n:
    if acceptable(candidate, candidate[0]):
      yield candidate[:]
  else:
    for e in range(2**n):
      if not accepted or e not in accepted and acceptable(candidate, e):
        candidate.append(e)
        accepted.add(e)
        yield from gray_code(n, candidate, accepted)
        accepted.remove(e)
        candidate.pop()

assert [0, 1, 3, 2] == next(gc := gray_code(2), None)
assert 7 == len(tuple(gc))

In [ ]:
def grey_code(n):
  if n == 1:
    yield from ((0, 1), (1, 0))
  for e in grey_code(n-1):
    yield e
  for e in reversed(grey_code(n-1)):
    yield e

!! **factorize**, gcd, **primality test** (is prime?), **next_prime** (probabilistic, exact), lcm, etc.

In [ ]:
def factorize(n, d=2):  # n: dividend, d: divisor
  if n==1:
    return tuple()
  elif n % d==0:
    return (d,) + factorize(n // d)
  elif n < d**2:
    return (n,)
  else:
    return factorize(n, 3 if d==2 else d+2)

def gcd(n, d):  # n: dividend, d: divisor.
  while d>0:
    n, d = d, n % d
  return n

def test_primality(n):
  if n in {2, 3}:
    return True
  if n==1 or n % 2==0:
    return False
  for d in range(3, int(sqrt(n))+1, 2):  # d: divisor, 3, 5, 7, ... (step=2)
    if n % d==0:
      return False
  return True

def next_prime(n, certainty=5):  # returns when the prob. exceeds 1-0.5 ** certainty.
  if n<4:
    return max(2, n)  # prime p >=2
  if n % 2==0:
    n += 1
  for p in range(n, 2 * n):  # prime p exists where n < p <2n-2 when n >1.
    for _ in range(certainty):
      a = 2 + randrange(p-3)  # 2 <= a <= p-2; 2+0 <= a <=2+p-4.
      if 1 != a**(p-1) % p:  # rabin miller's primality test.
        break
    else:
      return p

lcm = lambda a, b: a * b // gcd(a, b)
assert 36 == lcm(2 * 3**2, 2**2 * 3)  # LCM: 2**2 * 3**2.
assert [2, 3, 5, 7, 11] == [e for e in range(12) if test_primality(e)]
assert [2, 2, 2, 3, 5, 5, 7, 7, 11, 11] == [next_prime(n) for n in range(0, 10)]
assert (2, 2, 3) == factorize(12) and (2, 3, 7) == factorize(42)
assert (3, 3, 5) == factorize(45) and (3, 5, 5) == factorize(75)

In [ ]:
# @title ##### 1.1 Write a program to determine if a string has all unique characters. What if you cannot use additional data structures?
def uniq(s) -> bool:  # time: O(n), space(n)
  return all(v == 1 for v in histogram(s).values())

def histogram(iterable) -> dict:
  h = {}
  for e in iterable:
    h[e] = h.get(e, 0)+1
  return h

def uniqq(s) -> bool:  # time: o(n^2), space: o(1)
  for i_lhs, lhs in enumerate(s):
    for i_rhs in range(i_lhs+1, len(s)):
      if lhs == s[i_rhs]:
        return False
  return True

assert all([uniq(""), uniq("a"), uniq("ab"), not uniq("aa"), not uniq("aba")])
assert all([uniqq(""), uniqq("a"), uniqq("ab")])
assert not any([uniqq("aa"), uniqq("aba")])

In [ ]:
# @title ###### 1.2 Write a program to test if two strings are permutations of each other.
def anagram(s1, s2):
  if len(s1) != len(s2):
    return False
  h = Counter(s1)
  for e in s2:
    if h.get(e, 0) >0:
      h[e] -= 1
    else:
      return False
  return True

def anagram2(s1, s2):
  signature = lambda s: ''.join(sorted(s))
  return len(s1) == len(s2) and signature(s1) == signature(s2)

def anagram3(s1, s2):
  return Counter(s1) == Counter(s2)

assert anagram('', '') and anagram('a', 'a') and anagram('ab', 'ba')
assert anagram('aab', 'aba') and anagram('aabb', 'abab')
assert not anagram('a', '') and not anagram('', 'a')
assert not anagram('a', 'b') and not anagram('aa', 'ab')
assert anagram2('aab', 'aba') and anagram2('aabb', 'abab')
assert anagram3('aab', 'aba') and anagram3('aabb', 'abab')

In [ ]:
# @title ##### 1.3 Write a program to replace all spaces in a string with %20.
def escape_spaces(s):
  L, m = list(s), len(s)  # m: input length.
  resize(L, n := len(L)+2 * L.count(" "), " ")  # n: output length.
  w = n
  for r in range(m-1, -1, -1):  # i in [m-1, 0].
    buff = "%20" if L[r] == " " else [L[r]]
    w -= len(buff)
    L[w:w + len(buff)] = buff
  return "".join(L)

assert "a%20b%20c%20" == escape_spaces("a b c ")

In [ ]:
# @title ##### 1.4 Write a program to test if a string is a permutation of a palindrome.
palindormic = lambda s: sum(v % 2 for v in Counter(s.lower()).values()) <2
assert all([palindormic(""), palindormic("a"), palindormic("aa"), palindormic("aba")])
assert not any([palindormic("ab"), palindormic("abbc")])

!! **edit distance** 1.5 Write a program to test if two strings are one or zero edits away from each other (insert, delete, or replace).

In [ ]:
def edit(a, b):
  @lru_cache(maxsize=None)
  def prog(m, n):
    if m == 0 or n == 0:
      return (m + n, ((Edit.D,) * m if m >0 else (Edit.I,) * n))
    c = int(a[m-1] != b[n-1])  # edit cost: 1 or 0.
    cases = (
        recap(prog(m-1, n-1), c, ([Edit.M, Edit.R][c],)),
        recap(prog(m-1, n-0), 1, (Edit.D,)),
        recap(prog(m-0, n-1), 1, (Edit.I,)),
    )
    return min(cases, key=lambda cost_steps: cost_steps[0])

  return prog(len(a), len(b))

Edit = Enum("Edit", {"I": "Insert", "D": "Delete", "R": "Replace", "M": "Match"})
expected = (3, (Edit.R, Edit.M, Edit.M, Edit.M, Edit.R, Edit.M, Edit.I))
assert expected == edit("kitten", "sitting")
assert (3, (Edit.D,) * 3) == edit("cat", "")
assert (3, (Edit.I,) * 3) == edit("", "sit")

In [ ]:
# @title ##### 1.6 Write a method to compress a string using counts of repeated chars, e.g., aabcccccaaa becomes a2b1c5a3.
def compressed(s):
  s2, m, start = [], len(s), 0
  for stop in range(1, m+1):
    if stop == m or s[start] != s[stop]:
      s2.extend([s[start], str(stop - start)])
      start = stop
  return "".join(s2) if len(s2) < m else s

assert "a2b1c5a3" == compressed("aabcccccaaa")
assert "abcc" == compressed("abcc")
assert "abc" == compressed("abc")
assert "" == compressed("")

In [ ]:
# @title ##### 1.7 Given an image represented by an NxN matrix, write a program to rotate the image by 90 degrees; in-place, in O(1) space.
def rotate(g):
  n = len(g)
  for layer in range(n // 2):
    head, tail = layer, n-1 - layer
    for i in range(head, tail):
      top = g[layer][i]
      g[layer][i] = g[n-1 - i][head]  # left to top
      g[n-1 - i][head] = g[tail][n-1 - i]  # bottom to left
      g[tail][n-1 - i] = g[i][tail]  # right to bottom
      g[i][tail] = top  # top to right
  return g

g = [
    [1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10],
    [11, 12, 13, 14, 15],
    [16, 17, 18, 19, 20],
    [21, 22, 23, 24, 25],
]
expected = [
    [21, 16, 11, 6, 1],
    [22, 17, 12, 7, 2],
    [23, 18, 13, 8, 3],
    [24, 19, 14, 9, 4],
    [25, 20, 15, 10, 5],
]
assert expected == rotate(g)

In [ ]:
# @title ##### 1.8 Given an NxN matrix, write a program to set the entire row and column to 0 if an element has a value of 0.
def zero_out(g):
  m, n = len(g), len(g[0])
  rows, columns = set(), set()
  for r in range(m):
    for c in range(n):
      if g[r][c] == 0:
        rows.add(r)
        columns.add(c)
  for r in range(m):
    for c in range(n):
      if r in rows or c in columns:
        g[r][c] = 0
  return g

g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 0, 1, 2], [3, 4, 5, 6]]
assert [[1, 0, 3, 4], [5, 0, 7, 8], [0, 0, 0, 0], [3, 0, 5, 6]] == zero_out(g)

In [ ]:
# @title ##### 1.9 Given two strings, write a program to test if a string is a rotation of the other using isSubstring method.
is_rotated = lambda s, t: len(s) == len(t) and (s + s).find(t) > -1
assert all([is_rotated("x", "x"), is_rotated("xy", "yx"), is_rotated("xyz", "yzx")])
assert not is_rotated("xyz", "xyx")

In [ ]:
# @title ##### 2.1 Write a program to remove duplicates from an unordered linked list. What if you cannot use additional data structures?
def dedup_o_n_time(L):
  curr, seen = L, {}
  while curr:
    if curr.value in seen:
      pred.next = curr.next
    else:
      seen[curr.value], pred = True, curr
    curr = curr.next
  return L

def dedup_o_1_space(L):
  lhs = L
  while lhs:
    pred = lhs  # predecessor of RHS.
    while pred.next:
      if lhs.value == pred.next.value:
        pred.next = pred.next.next
      else:
        pred = pred.next
    lhs = lhs.next
  return L

L3 = SNode.from_values(1, 2, 3)
assert L3 == dedup_o_n_time(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_n_time(SNode.from_values(1, 2, 1, 2, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 1, 2, 3, 3))
assert L3 == dedup_o_1_space(SNode.from_values(1, 2, 1, 2, 3))

In [ ]:
# @title ##### 2.2 Implement an algorithm to find the k-th to last element of a singly linked list.
def last(L: SNode, k=0):
  p, pk = L, None  # pk points to the k-th last node when p reaches the end.
  for _ in range(k):
    if p is None:
      break
    p = p.next
  if p:
    pk = L
    while p.next:
      p, pk = p.next, pk.next
  return pk

L4 = SNode.from_values(*range(4))  # 0, 1, 2, 3
assert [3, 2, 0, None] == [(_ := last(L4, k)) and _.value for k in (0, 1, 3, 4)]

In [ ]:
# @title ##### 2.3 Given access only to a node, implement an algorithm to delete that node in the middle of a singly linked list.

In [ ]:
# @title ##### 2.4 Write a program to partition a linked list around a value of x, such that all nodes less than x come before all nodes greater than or equal to x.
def partition(L, x):
  def push(pool, curr):
    curr.next = pool
    return curr
  def last(curr):
    while curr.next:
      curr = curr.next
    return curr

  lt_x, eq_x, gt_x, = [None], [None], [None]  # 1-element containers.
  curr = L
  while curr:
    next_ = curr.next
    pool = lt_x if curr.value < x else gt_x if curr.value > x else eq_x
    pool[0] = push(pool[0], curr)
    curr = next_
  last(lt_x[0]).next, last(eq_x[0]).next = eq_x[0], gt_x[0]
  return lt_x[0]

L9 = SNode.from_values(9, 3, 8, 2, 5, 6, 1, 7, 4, 5)
assert [4, 1, 2, 3, 5, 5, 7, 6, 8, 9] == [e.value for e in partition(L9, 5)]

2.5. Given two single linked lists where each node has a single digit, write a program that sums them up.

In [ ]:
def sum_integer_strings(L, R):  # Sum two integers in strings.
  L, R = (list(reversed(e)) for e in (L, R))  # becomes a: addend and augend
  outcome = []
  tens = 0
  zero = ord("0")
  for i in range(1 + max(len(L), len(R))):
    ones = tens
    ones += sum(ord(a[i]) - zero for a in (L, R) if i < len(a))
    tens, ones = ones // 10, ones % 10
    outcome.append(chr(zero + ones))
  return "".join(reversed(outcome))

assert "1300" == sum_integer_strings("313", "987")

2.6 Write a program to test if a linked list is palindromic.

2.7 Given two linked lists that interacts by reference (not value), write a program to return the intersecting node.

2.8 Given a linked list, implement an algorithm which returns the node at the beginning of the loop. e.g., INPUT: a -> b -> c -> d -> e -> c, and OUTPUT: c.



3.1 How would you design and implement three stacks on a single array.

In [ ]:
# @title ###### 3.2 Design a stack that has **min** method that returns the minimum in addition to push and pop methods. Push, pop, and min should all operate in O(1) time.
class MinStack:
  def __init__(self):
    self.minimum, self.stack = None, []
  def push(self, e):
    if self.minimum is None or e <= self.minimum:
      self.stack.append(self.minimum)  # saves the previous minimum.
      self.minimum = e
    self.stack.append(e)
    return self
  def pop(self):
    if (e := self.stack.pop()) == self.minimum:
      self.minimum = self.stack.pop()
    return e

S = MinStack().push(2).push(3).push(2).push(1)
assert 1 == S.minimum and 1 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert 2 == S.minimum and 3 == S.pop()
assert 2 == S.minimum and 2 == S.pop()
assert S.minimum is None

3.3 Imagine a literal stack of plates. If the stack gets too high, it might topple. Therefore, in real life, we would likely start a new stack when the previous stack exceeds some threshold. Implement a data structure SetOfStacks that mimics this. SetOfStacks should be composed of several stacks and should create a new stack once the previous one exceeds capacity. SetOfStacks.push() and SetOfStacks.pop() should behave identically to a single stack (that is, pop() should return the same values as it would if there were just a single stack). Implement a function popAt(int index) which performs a pop operation on a specific sub-stack.

In [ ]:
# @title ##### 3.5 Design a queue using two stacks.
class Q:  # debug with repr: Q.__repr__ = __repr__
  def __init__(self, inbox=[], outbox=[]):
    self.inbox, self.outbox = inbox, outbox
  def offer(self, E):  # E: element.
    self.inbox.append(E)
    return self
  def poll(self):
    if not self.outbox:
      while self.inbox:
        self.outbox.append(self.inbox.pop())
    return self.outbox.pop() if self.outbox else None

(q := Q()).offer(1).offer(2)
assert 1 == q.poll() and q.offer(3) is q
assert (2, 3, None) == tuple(q.poll() for _ in range(3))

In [ ]:
# @title ##### 3.6 Write a program to sort a stack so the largest elements are at the top. You may use additional stacks to hold items, but you may not copy the elements into any other data structure such as an array. The stack supports the following operations: push, pop, peek, and isEmpty.
def sort(S):  # a = [1, 2, 9, 8, 3, 4]
  L = []
  while S:
    e = S.pop()
    while L and L[-1] > e:
      S.append(L.pop())
    L.append(e)
  S.extend(L)
  return S

assert [1, 2, 3, 4, 5] == sort([1, 2, 5, 3, 4])

3.7 An animal shelter holds only dogs and cats, and operations on a strictly "first in, first out" basis. People must adopt either the oldest (based on the arrival time) of all animals at the shelter, or they can select whether they would prefer a dog or a cat (and will receive the oldest animal of that type). They cannot select which speicific animal they would like. Create the data structures to maintain this system and implement operations such as enqueue, dequeueAny, dequeueDog, and dequeueCat. You may use the LinkedList data structure.

In [ ]:
# @title ##### def optimal_tour(g):
def optimal_tour(g):  # TSP http://www.youtube.com/watch?v=aQB_Y9D5pdw
  def popped(s, i):
    l = list(s)
    l.pop(i)
    return tuple(l)

  # g(v, set) is min([g[v, e] + g(e, set-{e}) for e in set])
  def mapped(v, s, memos={}):
    @lru_cache(maxsize=None)
    def computed(v, s):
      if s:
        return min(((w, g[v][w] + mapped(w, popped(s, i))[1])
                    for i, w in enumerate(s)
                    if g[v][w] != inf),
                   key=lambda e: e[1])
      else:
        return (None, inf) if g[v][0] is inf else (0, g[v][0])

    return computed(v, s)

  return mapped(0, tuple(range(1, len(g))))

g = [
    [0, 10, 15, 20],
    [5,  0,  9, 10],
    [6, 13,  0, 12],
    [8,  8,  9,  0]
]  # yapf:disable
assert (1, 35) == optimal_tour(g)

In [ ]:
# @title ##### Find the shortest paths between all pairs of vertices, [floyd-Warshall](https://en.wikipedia.org/wiki/floyd-Warshall_algorithm).
g = [  #
    [0, inf, -2, inf],
    [4, 0, 3, inf],
    [inf, inf, 0, 2],
     [inf, -1, inf, 0]
]  # yapf: disable
expected = [
    [0, -1, -2, 0],
    [4,  0,  2, 4],
    [5,  1,  0, 2],
    [3, -1,  1, 0]
]  # yapf: disable

def floyd_warshall(graph):  # https://en.wikipedia.org/wiki/floyd-Warshall_algorithm
  d = [r[:] for r in g]  # distance matrix: a deep copy of n x n matrix.
  for k in range(n := len(graph)):  # graph: n x n matrix.
    for i in range(n):
      for j in range(n):
        d[i][j] = min(d[i][j], d[i][k] + d[k][j])
  return d

assert expected == floyd_warshall(g)

!! **vertex-color, edge-color** Given a list of 1:1 meetings, find the minimum number of time slots for scheduling them all. No one can be in more than one meeting a time.

In [ ]:
def color_vertex(g):  # yield a tuple (chromatic index, color sequence).
  def backtrack(color_seq):
    if len(color_seq) == len(g):
      yield tuple(color_seq)
    else:
      for color in expand_out(color_seq):  # color c may be an old color, or a new one.
        color_seq.append(color)
        yield from backtrack(color_seq)
        color_seq.pop()
  def expandable(color_seq, c):  # suitable for an expansion within constraints?
    v = len(color_seq)  # v: new vertex, v0: old vertex, where v0 < v.
    return all([g[v][v0] == 0 or color_seq[v0] != c for v0 in range(v)])
  def expand_out(color_seq):  #
    assert len(color_seq) < len(g)
    yield from chain((c for c in set(color_seq) if expandable(color_seq, c)),
                     [max(color_seq)+1])

  yield from ((max(e)+1, e) for e in backtrack([0]))

# a0 - b1
# |    |
# c2 - d3 - e4
g = [
    [0, 1, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0],
    [0, 1, 1, 0, 1],
    [0, 0, 0, 1, 0],
]
# (3, (0, 1, 1, ...)): color: chromatic index 3, color sequence: (0, 1, 1, ...).
assert [(2, (0, 1, 1, 0, 1))] == minima(color_vertex(g), key=lambda e: e[0])

!! **iterate, riterate a binary search tree** using a stack; iterative inorder traversal without recursion.

In [ ]:
def iterate(tree):  # returns an iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.left
    else:
      yield (tree := stack.pop())
      tree = tree.right

def riterate(tree):  # returns a reverse iterator.
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      tree = tree.right
    else:
      yield (tree := stack.pop())
      tree = tree.left

L = BTree(3, BTree(2, BTree(1, BTree(0))))
R = BTree(8, BTree(5, None, BTree(7, BTree(6))), BTree(9))
t10 = BTree(4, L, R)
assert list(range(10)) == list(tree.key for tree in iterate(t10))
assert [9, 8] == list(e.key for e in islice(riterate(t10), 2))
assert [8, 7] == list(e.key for e in islice(riterate(t10), 1, 3))
assert [8, 6, 4] == list(e.key for e in islice(riterate(t10), 1, 6, 2))

!! **acyclic or cyclic** Find a cycle if one exists? Detect a deadlocked if one exists?

* undirected: it is a tree if the undirected graph is connected and has n - 1 edges for n vertices.
* directed: a directed acyclic graph has no back edges; a cyclic graph has back edges.

In [ ]:
def find_cycles(graph, directed=False):  # g: graph either directed, or undirected.
  def dfs_(v, entered, exited, directed, parent):
    if v not in entered:
      entered.add(v)
      for w in graph[v]:
        if w not in entered:
          yield from dfs_(w, entered, exited, directed, v)
        elif directed and w not in exited:  # a back edge, unless w is exited.
          yield (v, w)
        elif not directed and parent != w:  # a back edge, unless w is a parent.
          yield (v, w)
      exited.add(v)

  entered, exited = set(), set()
  yield from (cycles := (
      cycle  #
      for vertex, _ in enumerate(graph)  #
      for cycle in dfs_(vertex, entered, exited, directed, parent=None)))

"""graph:
A0 →  C2 → E4
↓  ↗  ↓   ↗
B1 .  D3"""
a, b, c, d, e = range(5)
g = [
  {b, c},  # from a
  {c},     # from b
  {d, e},  # from c
  {e},     # from d
  {}       # from e
]  # yapf:disable
assert not next(find_cycles(g, True), None)

g[a] = {b}
g[c] = {a, d, e}  # this change implants a cycle.
assert [(2, 0)] == list(find_cycles(g, True))

# undirected graph: A0 - B1 - C2
a, b, c = range(3)
g2 = [{b}, {a, c}, {b}]
assert not next(find_cycles(g2, False), None)

# undirected graph: A0 - B1 - C2 - A0
g2[a].add(c)
g2[c].add(a)
assert [(2, 0), (0, 2)] == list(find_cycles(g2, False))

!! **topological sort** 4.7 Given a set of projects and their dependencies. Find a build order that ensures each project is built only after its dependencies are built.

!! **reachable, connected components** 4.1 Given a directed graph, design an algorithm to find whether there is a route between two nodes.

In [ ]:
# E = Edge = namedtuple("Edge", "w v weight", defaults=[-1, 0])  # v to w edge.
E = Edge = namedtuple("Edge", "w v", defaults=[-1])  # v to w edge.
E.replace = E._replace
OP = OpCode = IntFlag("OpCode", "ENTER CROSS EXIT")

def dfs(graph, initial_edge, entered=None):  # initial_edge, also called start_edge.
  if entered is None:
    entered = set()
  if (v0 := initial_edge.w) not in entered:
    entered.add(v0)
    yield OP.ENTER, initial_edge
    for edge in graph[v0] or []:
      yield OP.CROSS, (edge := edge._replace(v=v0))
      yield from dfs(graph, edge, entered)
    yield OP.EXIT, initial_edge

def topological_sort(graph):
  entered = set()
  for v, _ in enumerate(graph):
    if v not in entered:
      yield from (
          edge.w for op, edge in dfs(graph, Edge(v, -1), entered) if op == OP.EXIT)

def can_reach(graph, source, sink) -> bool:
  return any(edge.w == sink for op, edge in dfs(graph, Edge(source)) if op == OP.CROSS)

# graph: .  .  D3 ⇾ H7
#  .  .  .  .  ↑
#   ┌───────── B1 ⇾ F5
#   ↓  .  .  . ↑  .  ↑
#   J9 ⇽ E4 ⇽ A0 ⇾ C2 ⇾ I8
#   ↓
#   G6
graph = [  #
    [Edge(1), Edge(2), Edge(4), Edge(6)],  # 1, 2, 4, and 6 depend on 0.
    [Edge(3), Edge(5), Edge(9)],  # 3, 5, and 9 depend on 1.
    [Edge(5), Edge(8)],  # 5, 8 depend on 2.
    [Edge(7)],  # 7 depend on 3.
    [Edge(9)],  # 9 depend on 4.
    *([[]] * 5),
]
expected = (7, 3, 5, 9, 1, 8, 2, 4, 6, 0)
assert deque(expected) == deque(topological_sort(graph))
assert all(can_reach(graph, source, 9) for source in (0, 1, 4))
assert not any(can_reach(graph, source, 0) for source in range(4))

In [ ]:
# @title ##### 4.2 Given a sorted (increasing order) array, design an algorithm to create a binary search tree with the minimal height.
# tree:  2
#      /  \
#     1    3
#           4
t4 = BTree.from_values((1, 2, 3, 4))
assert "BTree(2, BTree(1), BTree(3, None, BTree(4)))" == repr(t4)

In [ ]:
# @title ##### 4.3 Given a binary tree, design an algorithm to creates a linked list of all the nodes at each depth, e.g., if you have a tree with depth D, you will have D linked lists.
def to_d_linked_lists(tree):  # tree of depth d.
  LL = []  # output: a list of lists.
  q = [tree]
  while q:
    p = []
    for i, e in enumerate(q):
      for c in (e.left, e.right):
        if c:
          p.append(c)
      e.left = None if i == 0 else q[i - 1]
      e.right = q[i + 1] if i < len(q) else None
    LL.append(q[0])
    q = p
  return LL

In [ ]:
# @title ##### 4.4 Implement a function to check if a binary tree is balanced. For the purposes of this question, a balanced tree is defined to be a tree such that the heights of two subtrees of any node never differ by more than one.
# tree: a
# . .  /   \
# .   b  .  f
#   c  e   g
#  d
c = BTree("c", BTree("d"))
e = BTree("e")
b = BTree("b", c, e)
a = BTree("a", b, BTree("f"))

def is_balanced(tree):  # returns (balanced, height)
  if tree is None:
    return (True, -1)
  L, R = is_balanced(tree.left), is_balanced(tree.right)
  b = L[0] and R[0] and abs(L[1] - R[1]) <2
  h = 1 + max(L[1], R[1])
  return (b, h)

assert not is_balanced(a)[0]
a.right.left = BTree("g")
assert (True, 3) == is_balanced(a)  # balanced, height: 3.

In [ ]:
# @title ##### 4.5 Implement a function to check if a binary tree is a binary search tree.
def iterate_(tree):
  if tree:
    yield from chain(iterate_(tree.left), (tree,), iterate_(tree.right))

def is_ordered(tree):
  if pred := next(it := iterate_(tree), None):
    for e in it:
      if e.value < pred.value:
        return False
      else:
        pred = e
  return True

def ordered(tree):  # returns (ordered, min, max)
  if tree is None:
    return (True, None, None)
  L, R = ordered(tree.left), ordered(tree.right)
  O = (L[0] and R[0]
       and (L[2] is None or L[2] <= tree.value)
       and (R[1] is None or R[1] >= tree.value))  # yapf:disable
  min_ = tree.value if L[1] is None else L[1]
  max_ = tree.value if R[2] is None else R[2]
  return (O, min_, max_)

b7 = BTree.from_values([1, 2, 3, 4, 5, 6, 7])
assert is_ordered(b7)
assert ordered(b7)[0]
assert not ordered(BTree.from_values((1, 2, 3, 4, 0, 6, 7)))[0]

In [ ]:
# @title ##### 4.6 Design an algorithm to find the next node (i.e., successor in-order) of a node in a binary search tree. You may assume that each node has a link to its parents.
# tree: .  z
#  .  .  u
#  .  .  . v
#  .  .  .   y
#  .  .  . x
#  .  .  w
w = BTree("w")
x = BTree("x", w)
y = BTree("y", x)
v = BTree("v", None, y)
u = BTree("u", None, v)
z = BTree("z", u)

def set_parent(tree):  # set value fields on the children/descendants.
  tree._parent = None
  stack = []
  while tree or stack:
    if tree:
      stack.append(tree)
      if tree.left:
        tree.left._parent = tree
      tree = tree.left
    else:
      tree = stack.pop()
      if tree.right:
        tree.right._parent = tree
      tree = tree.right

def succ(node):
  if node is None:
    raise RuntimeError("'node' must be non-null.")
  if node.right:
    node = node.right
    while node.left:
      node = node.left
    return node
  else:
    while node._parent and node == node._parent.right:
      node = node._parent
    return node._parent

set_parent(z)
assert (v, w, x, y, z, None) == tuple(succ(tree) for tree in (u, v, w, x, y, z))

In [ ]:
# @title ##### 4.8 Design an algorithm to find the first common ancestor of the nodes in a binary tree. Avoid storing additional nodes in a data structure. Note: this is not necessarily a binary search tree.
def is_subtree(tree, subtree):
  return (
    tree == subtree
    or tree
    and (is_subtree(tree.left, subtree) or is_subtree(tree.right, subtree))
  )

def lowest_common_ancestor(tree, p, q):
  if is_subtree(tree, p) and is_subtree(tree, q):
    while tree:
      if tree is p or tree is q:
        return tree
      p_on_left = is_subtree(tree.left, p)
      q_on_left = is_subtree(tree.left, q)
      if p_on_left != q_on_left:
        return tree
      tree = tree.left if p_on_left else tree.right

def lowest_common_ancestor2(tree, p, q):  # returns (count, LCA)
  if tree is None:
    return (None, 0)
  count = (p, q).count(tree)
  if count == 2:
    return (tree, 2)
  l = lowest_common_ancestor2(tree.left, p, q)
  if l[1] == 2:
    return l
  r = lowest_common_ancestor2(tree.right, p, q)
  if r[1] == 2:
    return r
  count += l[1] + r[1]
  return (tree if count == 2 else None, count)

# tree:.   a
#  .  .  /   \
#  .   b  .   f
#  .  c  e  g
#   d
d = BTree("d")
c = BTree("c", d)
e = BTree("e")
b = BTree("b", c, e)
a = BTree("a", b, BTree("f", BTree("g")))
assert is_subtree(a, a)
assert is_subtree(a, a.left) and is_subtree(a, a.left.right)
assert is_subtree(a, a.right) and is_subtree(a, a.right.left)
assert b == lowest_common_ancestor(a, d, e)
assert b == lowest_common_ancestor2(a, d, e)[0]

!! **parse, diameter** Given pre-order and in-order tree traversal sequences, parse a binary tree, and compute the diameter of the tree.

In [ ]:
def diameter(tree):  # return (diameter, height).
  if tree is None:
    return (0, 0)
  L = diameter(tree.left)
  R = diameter(tree.right)
  dia = max(L[0], R[0], 1 + L[1] + R[1])
  return (dia, height := 1 + max(L[1], R[1]))

def parse(preorder, inorder):
  if preorder:
    value = preorder[0]
    m = inorder.index(value)
    return BTree(v, parse(preorder[1:m+1], inorder[:m]),
                 parse(preorder[m+1:], inorder[m+1:]))

# tree input:   a
#             b
#          c    f
#           d     g
#            e
tree = parse("abcdefg", "cdebfga")
assert (6, 5) == diameter(tree)

In [ ]:
# @title ##### 4.9 Given a binary search tree that was reconstructed from an array of integers, find all the possible arrays that could have led to this binary search tree.
def weave(L, R):
  @lru_cache(maxsize=None)
  def prog(m, n):
    if m == 0 or n == 0:
      yield L[:m] + R[:n]
    else:
      yield from chain(
          (e + [L[m-1]] for e in prog(m-1, n)),
          (e + [R[n-1]] for e in prog(m, n-1)),
      )

  yield from prog((m := len(L)), (n := len(R)))

def destruct(tree: BTree):
  if tree is None:
    yield []
  else:
    yield from ([tree.value] + e
                for L in destruct(tree.left)
                for R in destruct(tree.right)
                for e in weave(L, R))

T5 = BTree(4, BTree(1, None, BTree(3, BTree(2))), BTree(5))
expected = ([4, 5, 1, 3, 2], [4, 1, 5, 3, 2], [4, 1, 3, 5, 2], [4, 1, 3, 2, 5])
assert expected == tuple(destruct(T5))

In [ ]:
# @title ##### 4.10 You have two very large binary trees: T1 with millions of nodes, and T2 with hundreds of nodes. Design an algorithm to decide if T2 is a subtree of T1. A tree T2 is a subtree of T1 if there exists a node in T1 such that the subtree of n is identical to T2. i.e., if you cut off the tree at node n, the two trees would be identical.
def contains(tree, subtree):
  return (
    subtree is None
    or starts_with(tree, subtree)
    or (tree and (contains(tree.left, subtree) or contains(tree.right, subtree)))  # fmt: skip
  )  # yapf:disable

def starts_with(tree, subtree):
  return (
    subtree is None
    or tree
    and tree.value == subtree.value
    and starts_with(tree.left, subtree.left)
    and starts_with(tree.right, subtree.right)
  )  # yapf:disable

assert contains(a, None) and contains(a, a)
assert contains(a, BTree("b", BTree("c")))
assert contains(a, BTree("b", None, BTree("e")))
assert contains(a, BTree("b", BTree("c"), BTree("e")))

!! **seekable binary search tree** 4.11 Design a binary search tree capable of selecting a random node with equal probability. Operations such as inserting, deleting, finding, and randomly selecting a node should run in sublinear time.

* Seekable binary search tree
  * bisect method yields bisecting tree nodes from the leaf to the root.
  * getitem method returns the leaf node from the bisect, or none, if not found.
  * setitem method get the leaf node from bisect, and makes a node of a new key.
  * seek method returns the node, when `index` equals to the size of L sub-tree.

In [ ]:
class BTree:  # seekable binary search tree.
  def __init__(self, key=None, left=None, right=None, value=None):
    lr = (c if c is None or isinstance(c, BTree) else BTree(c) for c in (left, right))
    self.key, self.left, self.right, self.value_ = key, *lr, value
    self._size = sum(bool(c) and c._size for c in (self.left, self.right))+1

  @property
  def value(self):
    return self.key if self.value_ is None else self.value_
  #
  def bisect(self, key):
    if key < self.key and self.left:
      yield from self.left.bisect(key)
    elif key > self.key and self.right:
      yield from self.right.bisect(key)
    yield self
  #
  def __getitem__(self, key):
    return (b if (b := next(self.bisect(key))).key == key else None)
  #
  def __setitem__(self, key, value):
    it = more_itertools.peekable(self.bisect(key))
    if (tree := it.peek()).key == key:
      tree.value_ = value
    else:
      if tree.key > key:
        tree.left = BTree(key, value=value)
      else:
        tree.right = BTree(key, value=value)
      for tree in it:  # Recalculate tree._size in a bottom-up fashion.
        tree._size = 1 + sum(c._size for c in (tree.left, tree.right) if c)
  #
  def seek(self, index: int):
    L_size = (L := self.left) and L._size or 0
    if index > L_size:
      return self.right.seek(index-1 - L_size) if self.right else None
    elif index < L_size:
      return self.left.seek(index) if self.left else None
    else:
      return self

# tree:   5
#       / . \
#      2  .  9
#       3 . 7
# . . . .    8
BTree.__repr__ = __repr__
b6 = BTree(5, BTree(2, None, 3), BTree(9, BTree(7, None, 8)))
b6[6] = None  # value property returns key, when value_ field is None.
assert (3, 5, 6) == tuple(b6.seek(i).value for i in range(1, 4))

!! **sequence sum** Given a binary tree of integers, find paths that sums up to a value. A path can start at any subtrees. e.g., paths (3, -1, 2), and (-1, 3, -1, 3) that sum up to 4 given this tree.
```
tree: -1
       \
        3
       /  \
     -1    2
     / \
    2   3
```

In [ ]:
def find_seq_sum(tree, seq_sum, cumsum_seq=[0], cumsum_indices=None):
  if tree:  # diff works like np.diff, that reverses np.cumsum.
    diff = lambda cumsum_seq: tuple(q - p for p, q in pairwise(cumsum_seq))
    cumsum_seq.append(cumsum := cumsum_seq[-1] + tree.value)
    cumsum_indices = cumsum_indices or defaultdict(list, {seq_sum: [0]})
    cumsum_indices[cumsum + seq_sum].append(len(cumsum_seq)-1)
    if cumsum_indices[cumsum]:
      yield from (diff(cumsum_seq[i:]) for i in cumsum_indices[cumsum])
    yield from chain(
        find_seq_sum(tree.left, seq_sum, cumsum_seq, cumsum_indices),
        find_seq_sum(tree.right, seq_sum, cumsum_seq, cumsum_indices))
    cumsum_seq.pop()
    cumsum_indices[cumsum + seq_sum].pop()

tree = BTree(-1, None, BTree(3, BTree(-1, BTree(2), BTree(3)), BTree(2)))
assert ((3, -1, 2), (-1, 3, -1, 3), (-1, 3, 2)) == tuple(find_seq_sum(tree, 4))

In [ ]:
# @title ##### 5.2 Given a double-precision floating-point number between 0 and 1, output its binary representation. If the exact binary representation requires more than 32 bits, print an error message.
def float_to_str(f: float) -> str:
  if f <= 0 or f >= 1:
    return "ERROR"
  L = list(".")
  while f > 0:
    if len(L) > 31:
      return "ERROR"
    f = round(2 * f, 6)
    if f >= 1:
      L.append("1")
      f -= 1
    else:
      L.append("0")
  return "".join(L)

assert ".011" == float_to_str(0.375)
assert ".101" == float_to_str(0.625)

In [ ]:
# @title ##### 5.3 Given an integer, write a program to find the length of the longest sequence of 1s after flipping exactly one 0 to 1.

!! **successor, next k-length subsequence, nCk (n choose k)** 5.4 Get the predecessor and successor integers with the same number of bits (1s).

In [ ]:
def successor(n: int) -> int:
  # count 0s, and 1s
  m = n.bit_length()
  count0 = next((c0 for c0 in range(m) if n & (1 << c0)), 0)  # c0: no. 0s.
  count1 = next((c1 for c1 in range(count0, m) if n & (1 << c1) == 0), m) - count0
  return n + (1 << count0) + (1 << count1-1)-1

def predecessor(n: int) -> int:
  # count 1s and 0s.
  m = n.bit_length()
  count1 = next((c1 for c1 in range(m) if n & (1 << c1) == 0), m)  # c1: no. 1s.
  count0 = next((c0 for c0 in range(count1, m) if n & (1 << c0)), 0) - count1
  return n - (1 << count1) - ((1 << count0-1)-1)

pred, succ = 256+16+8+4, 256+32+2+1
assert succ == successor(pred) and pred == predecessor(succ)

5.5 What does this code do? (n > 0 and n & (m-1) == 0)

5.6 Write a program to get the Hamming distance between two integers -- the number of positions at which the corresponding bits are different. `(lhs ^ rhs).bit_count()`.


5.7 Write a program to swap odd and even bits in an integer with as few instructions as possible.  
`swap_bits = lambda n: (n & 0xAAAAAAAA) >> 1 | (n & 0x55555555) << 1`

In [ ]:
# @title ##### 5.8 Design an algorithm to draw a horizontal line on a monochrome screen, where the screen is represented as a byte array with each byte storing 8 pixels. The function signature looks like: draw_line(screen: bytes[], width, x1, x2, y: int)